收益率分析的核心接口层，提供了对 Series 和 DataFrame 收益率数据的高级分析功能。

# 类继承关系
```mermaid
classDiagram
    class GenericAccessor
    class ReturnsAccessor
    class GenericSRAccessor
    class ReturnsSRAccessor
    class GenericDFAccessor
    class ReturnsDFAccessor

    %% 继承关系
    GenericAccessor <|-- ReturnsAccessor
    GenericSRAccessor <|-- ReturnsSRAccessor
    ReturnsAccessor <|-- ReturnsSRAccessor
    ReturnsAccessor <|-- ReturnsDFAccessor
    GenericDFAccessor <|-- ReturnsDFAccessor
```

# class ReturnsAccessor(GenericAccessor)

## `__init__`
参数
- `obj` (tp.SeriesFrame): 收益率数据，Series或DataFrame
- `benchmark_rets` (tp.Optional[tp.ArrayLike]): 基准收益率
  - None: 不使用基准，相关方法会报错或跳过
  - Series/DataFrame: 会自动广播匹配obj的维度
- `year_freq` (tp.Optional[tp.FrequencyLike]): 年化频率
  - None: 从全局设置获取，如'252D'表示252个交易日
  - 支持pandas频率字符串格式
- `defaults` (tp.KwargsLike): 默认参数覆盖
  - 可设置risk_free、window、ddof等参数
- `**kwargs`: 传递给GenericAccessor的其他参数

```python
    def __init__(self,
                 obj: tp.SeriesFrame,
                 benchmark_rets: tp.Optional[tp.ArrayLike] = None,
                 year_freq: tp.Optional[tp.FrequencyLike] = None,
                 defaults: tp.KwargsLike = None,
                 **kwargs) -> None:
        GenericAccessor.__init__(
            self,
            obj,
            benchmark_rets=benchmark_rets,
            year_freq=year_freq,
            defaults=defaults,
            **kwargs
        )

        if benchmark_rets is not None:
            benchmark_rets = broadcast_to(benchmark_rets, obj)
        self._benchmark_rets = benchmark_rets
        self._year_freq = year_freq
        self._defaults = defaults
```

# class ReturnsSRAccessor(ReturnsAccessor, GenericSRAccessor)
专门处理 `Series` 类型收益率数据的访问器类。

注意：
```python
@register_series_vbt_accessor('returns')
class ReturnsSRAccessor(ReturnsAccessor, GenericSRAccessor): ...
```
参考[root_accessors.ipynb](../root_accessors.ipynb)：
- 对于一个 `Series` 对象 `obj`，显然 `obj.vbt.returns` 相当于 `ReturnsSRAccessor(Vbt_SRAccessor(obj))`

## `__init__`
```python
    def __init__(self,
                 obj: tp.Series,
                 benchmark_rets: tp.Optional[tp.ArrayLike] = None,
                 year_freq: tp.Optional[tp.FrequencyLike] = None,
                 defaults: tp.KwargsLike = None,
                 **kwargs) -> None:
        GenericSRAccessor.__init__(self, obj, **kwargs)
        ReturnsAccessor.__init__(
            self,
            obj,
            benchmark_rets=benchmark_rets,
            year_freq=year_freq,
            defaults=defaults,
            **kwargs
        )
```

## `__init__`
```python
    @property
    def qs(self):
        from vectorbt.returns.qs_adapter import QSAdapter
        return QSAdapter(self)
```

# class ReturnsDFAccessor(ReturnsAccessor, GenericDFAccessor)
专门处理 `DataFrame` 类型收益率数据的访问器类。

注意：
```python
@register_dataframe_vbt_accessor('returns')
class ReturnsDFAccessor(ReturnsAccessor, GenericDFAccessor): ...
```
参考[root_accessors.ipynb](../root_accessors.ipynb)：
- 对于一个 `Series` 对象 `obj`，显然 `obj.vbt.returns` 相当于 `ReturnsDFAccessor(Vbt_DFAccessor(obj))`

## `__init__`
```python
    def __init__(self,
                 obj: tp.Frame,
                 benchmark_rets: tp.Optional[tp.ArrayLike] = None,
                 year_freq: tp.Optional[tp.FrequencyLike] = None,
                 defaults: tp.KwargsLike = None,
                 **kwargs) -> None:
        GenericDFAccessor.__init__(self, obj, **kwargs)
        ReturnsAccessor.__init__(
            self,
            obj,
            benchmark_rets=benchmark_rets,
            year_freq=year_freq,
            defaults=defaults,
            **kwargs
        )
```